In [1]:
import pandas as pd 
from pathlib import Path

In [2]:
orders710 = pd.read_csv('../../orderlevel_updated/tier_merged/tiers_merged_0710-0713.csv')

In [3]:
orders710.head(5)

,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,basket_size,...,total_user_waiting_time_second,order_status,date,region::multi-filter,avg_rain,Rain_Level,Rider Group,Start Date,End Date,Incentive
0,SURIN,LMD00BZBZ,2025-07-10 10:27:54.432,ก๋วยเตี๋ยวถนอมมิตร2 (Thanommit Noodles 2),Noodles,LMF-250710-408959062,True,13.50,3.0,30.0,...,687,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,Streak
1,SURIN,LMD00BZBZ,2025-07-10 10:32:29.887,ตำซี๊ด&อาหารตามสั่ง (Tum Zeed & Made-to-Order ...,Dessert,LMF-250710-161870697,True,17.80,3.0,95.0,...,1223,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,Streak
2,SURIN,LMD00BZBZ,2025-07-10 10:47:04.631,เฉิงกง ติ่มซำ (Chenggong Dim Sum) -,Dim Sum,LMF-250710-166811972,True,12.66,3.0,230.0,...,2458,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,Streak
3,SURIN,LMD00BZBZ,2025-07-10 10:56:22.791,Café Amazon - SD3192 (คาเฟ่ อเมซอน - SD3192) ส...,Café/Coffee Shop,LMF-250710-400808907,True,12.37,3.0,120.0,...,1180,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,Streak
4,SURIN,LMD00BZBZ,2025-07-10 11:09:04.244,Café Amazon - DD1655 (คาเฟ่ อเมซอน - DD1655) ป...,Café/Coffee Shop,LMF-250710-858627724,True,14.30,3.0,245.0,...,2035,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,Streak


In [4]:
orders710.columns

Index(['region', 'driver_id', 'assign_created_at', 'restaurant_name',
       'restaurant_category', 'order_id', 'is_driver_accept', 'wage',
       'on_top_fare', 'basket_size', 'drop_off_distance',
       'distance_rider_to_restaurant', 'placed_to_match_second',
       'rider_matched_to_arrived_restaurant_second',
       'rider_waiting_time_in_restaurant_second',
       'delivery_time_to_user_second', 'total_user_waiting_time_second',
       'order_status', 'date', 'region::multi-filter', 'avg_rain',
       'Rain_Level', 'Rider Group', 'Start Date', 'End Date', 'Incentive'],
      dtype='object')

In [5]:
orders710[[       'rider_matched_to_arrived_restaurant_second',
       'rider_waiting_time_in_restaurant_second',
       'delivery_time_to_user_second', 'total_user_waiting_time_second',
]]

,rider_matched_to_arrived_restaurant_second,rider_waiting_time_in_restaurant_second,delivery_time_to_user_second,total_user_waiting_time_second
0,385,52,249,687
1,220,17,385,1223
2,261,1087,444,2458
3,707,24,156,1180
4,228,267,363,2035
...,...,...,...,...
446839,371,686,374,2204
446840,530,50,460,1213
446841,182,1246,221,2372
446842,1106,44,263,1709


In [6]:
csv_files = sorted(Path('../daily_productivity/daily_productivity_results').glob('*.csv'))
output_dir = Path('./daily_guarantee_results')
output_dir.mkdir(parents=True, exist_ok=True)

# display all files
for i, file in enumerate(csv_files, 1):
    print(f"{i}. {file.name}")
    
# Process each file
for i, csv_file in enumerate(csv_files, 1):
    print(f"\n[{i}/{len(csv_files)}] Processing: {csv_file.name}")
    print("-" * 80)
    
    df = pd.read_csv(csv_file)

    df['assign_created_at'] = pd.to_datetime(df['assign_created_at'])

    df = df.sort_values('assign_created_at')

    # Filter only accepted orders for counting 
    accepted_mask = df['is_driver_accept'] == True
    
    # Time per order 
    df['time_per_order_sec'] = df['rider_waiting_time_in_restaurant_second'] + df['delivery_time_to_user_second']

    # Total seconds worked each day (0 if not accepted)
    df['time_worked_sec'] = 0
    df.loc[accepted_mask, 'time_worked_sec'] = df.loc[accepted_mask, 'time_per_order_sec'] 
    
    # Extract date and hour
    df['date'] = df['assign_created_at'].dt.date
    df['hour'] = df['assign_created_at'].dt.hour
    
    # Cumulative sum per driver per date
    df['daily_cumulative_time_sec'] = df.groupby(['driver_id', 'date'])['time_worked_sec'].cumsum() 

    # Masks for time windows (10:00-12:59 or 17:00-19:59)
    time_window_mask = ((df['hour'] >= 10) & (df['hour'] <= 12)) | ((df['hour'] >= 17) & (df['hour'] <= 19))
    df['time_worked_in_window_sec'] = 0

    # Calculate time worked for orders in the window 
    df.loc[time_window_mask, 'time_worked_in_window_sec'] = df.loc[time_window_mask, 'time_worked_sec']
    
    # Running sum within each driver per day
    df['window_cumulative_sec'] = df.groupby(['driver_id', 'date'])['time_worked_in_window_sec'].cumsum()

    # Convert to hours 
    df['daily_cumulative_time_hr'] = df['daily_cumulative_time_sec'] / 3600
    df['window_cumulative_hr'] = df['window_cumulative_sec'] / 3600

    # Save to csv 
    output_path = output_dir / f'daily_guarantee_{csv_file.name}'
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

1. daily_productivity_tiers_merged_0710-0713.csv
2. daily_productivity_tiers_merged_0714-0716.csv
3. daily_productivity_tiers_merged_0717-0720.csv
4. daily_productivity_tiers_merged_0721-0725.csv
5. daily_productivity_tiers_merged_0726-0727.csv
6. daily_productivity_tiers_merged_0728-0801.csv
7. daily_productivity_tiers_merged_0802-0806.csv
8. daily_productivity_tiers_merged_0807-0810.csv
9. daily_productivity_tiers_merged_0807-0811.csv
10. daily_productivity_tiers_merged_0811-0815.csv
11. daily_productivity_tiers_merged_0816-0820.csv
12. daily_productivity_tiers_merged_0821-0825.csv
13. daily_productivity_tiers_merged_0826-0829.csv

[1/13] Processing: daily_productivity_tiers_merged_0710-0713.csv
--------------------------------------------------------------------------------
Saved: daily_guarantee_results/daily_guarantee_daily_productivity_tiers_merged_0710-0713.csv

[2/13] Processing: daily_productivity_tiers_merged_0714-0716.csv
-----------------------------------------------------

In [7]:
guarantee_710 = pd.read_csv('./daily_guarantee_results/daily_guarantee_daily_productivity_tiers_merged_0710-0713.csv')

In [8]:
guarantee_710

,Unnamed: 0,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,...,Incentive,daily_order_count,time_per_order_sec,time_worked_sec,hour,daily_cumulative_time_sec,time_worked_in_window_sec,window_cumulative_sec,daily_cumulative_time_hr,window_cumulative_hr
0,160727,UDONTHANI,LMDCSO52Z,2025-07-10 00:00:00.969,กะเพราทะลุฟ้า (Kapraotalufha) -,À La Carte,LMF-250709-194339998,True,20.29,0.0,...,Daily Productivity,1,472,472,0,472,0,0,0.131111,0.000000
1,34987,HAT YAI,LMD2P96XU,2025-07-10 00:00:11.812,Akita ชานมไข่มุก (Akita Bubble Tea) ตลาดศรีตรั...,Cafe,LMF-250710-967446340,True,22.00,0.0,...,Active Bonus,1,792,792,0,792,0,0,0.220000,0.000000
2,86388,SISAKET,LMD6ZOP8E,2025-07-10 00:00:26.512,ครัวคุณโต้ง อาหาร/เครื่องดื่ม (Krue.kun.tong) ...,Fastfood,LMF-250710-357336867,True,17.00,2.0,...,Daily Productivity,1,612,612,0,612,0,0,0.170000,0.000000
3,251286,KHONKAEN,LMDK8T3XZ,2025-07-10 00:00:27.543,เจียวกะพี่หมี ( หอพักสุทธิลักษณ์ ตึกA ),Rice Dish,LMF-250710-728920242,True,19.00,0.0,...,Streak,1,339,339,0,339,0,0,0.094167,0.000000
4,277399,CHIANGRAI,LMDMG4ZNH,2025-07-10 00:01:01.135,สถานีแจ่วฮ้อน MFU,Thai,LMF-250710-664531507,False,48.40,0.0,...,No Incentive,0,0,0,0,0,0,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446839,43748,KHONKAEN,LMD3D5FPY,2025-07-13 23:59:30.971,29 บุฟเฟ่ต์เครป Buffet Crepe By I Crepe (29Buf...,Dessert,LMF-250713-733548548,False,20.05,3.0,...,Streak,10,902,0,23,12318,0,4286,3.421667,1.190556
446840,409903,KANCHANABURI,LMDX2MLLG,2025-07-13 23:59:45.055,ลุงเต่าก๋วยเตี๋ยว+ส้มตำ (Uncle Tao) 1,Noodles,LMF-250713-893573106,True,20.00,3.0,...,No Incentive,20,547,547,23,14812,0,4834,4.114444,1.342778
446841,291390,NAKHONPATHOM,LMDNOGOA7,2025-07-13 23:59:47.832,นครปังนมสด กำแพงแสน,Dessert,LMF-250713-865985727,True,25.43,2.0,...,No Incentive,21,1812,1812,23,21191,0,5263,5.886389,1.461944
446842,312440,HAT YAI,LMDPBNV24,2025-07-13 23:59:57.488,Tanji Ramen สาขาจุติ,Japanese,LMF-250713-567050464,True,18.71,0.0,...,Active Bonus,20,775,775,23,17108,0,5705,4.752222,1.584722


In [9]:
guarantee_710[guarantee_710['driver_id'] == 'LMDCSO52Z']

,Unnamed: 0,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,...,Incentive,daily_order_count,time_per_order_sec,time_worked_sec,hour,daily_cumulative_time_sec,time_worked_in_window_sec,window_cumulative_sec,daily_cumulative_time_hr,window_cumulative_hr
0,160727,UDONTHANI,LMDCSO52Z,2025-07-10 00:00:00.969,กะเพราทะลุฟ้า (Kapraotalufha) -,À La Carte,LMF-250709-194339998,True,20.29,0.0,...,Daily Productivity,1,472,472,0,472,0,0,0.131111,0.000000
23335,160728,UDONTHANI,LMDCSO52Z,2025-07-10 10:59:50.639,Five Star (ห้าดาว) ดูโฮม อุดรธานี,Barbeque/Grill,LMF-250710-419574625,False,24.00,5.0,...,Daily Productivity,1,0,0,10,472,0,0,0.131111,0.000000
26735,160729,UDONTHANI,LMDCSO52Z,2025-07-10 11:16:01.800,ครัวพุงกาง (Pung Kang Kitchen),À La Carte,LMF-250710-984283443,True,19.00,5.0,...,Daily Productivity,2,981,981,11,1453,981,981,0.403611,0.272500
28080,160730,UDONTHANI,LMDCSO52Z,2025-07-10 11:22:21.890,ตำแซ่บอุดรธานี 1 (หน้าราชภัฏอุดร) (Tam Zaab Ud...,North East,LMF-250710-802551523,False,43.92,5.0,...,Daily Productivity,2,1355,0,11,1453,0,981,0.403611,0.272500
28248,160731,UDONTHANI,LMDCSO52Z,2025-07-10 11:23:11.190,อาหารตามสั่งครัวออโต้ (กะเพราแท้) (Auto Kitche...,À La Carte,LMF-250710-730251463,True,19.00,5.0,...,Daily Productivity,3,229,229,11,1682,229,1210,0.467222,0.336111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443633,160877,UDONTHANI,LMDCSO52Z,2025-07-13 22:00:52.190,กะเพราทะลุฟ้า (Kapraotalufha) -,À La Carte,LMF-250713-697795162,True,24.00,5.0,...,Daily Productivity,36,1139,1139,22,27922,0,13835,7.756111,3.843056
444735,160878,UDONTHANI,LMDCSO52Z,2025-07-13 22:29:53.257,is it ลาบยโส,North East,LMF-250713-685190429,True,24.00,0.0,...,Daily Productivity,37,769,769,22,28691,0,13835,7.969722,3.843056
445260,160879,UDONTHANI,LMDCSO52Z,2025-07-13 22:45:50.102,ต.กระเพราระเบิด (Spicy Lab Udon Made-to-Order ...,À La Carte,LMF-250713-810260717,True,24.00,0.0,...,Daily Productivity,38,328,328,22,29019,0,13835,8.060833,3.843056
446217,160880,UDONTHANI,LMDCSO52Z,2025-07-13 23:22:47.923,ณำนารา - Namnara,Café/Coffee Shop,LMF-250713-797684090,True,24.00,2.0,...,Daily Productivity,39,701,701,23,29720,0,13835,8.255556,3.843056


In [10]:
guarantee_710[(guarantee_710['driver_id'] == 'LMDCSO52Z') & (guarantee_710['date'] == '2025-07-10')]

,Unnamed: 0,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,...,Incentive,daily_order_count,time_per_order_sec,time_worked_sec,hour,daily_cumulative_time_sec,time_worked_in_window_sec,window_cumulative_sec,daily_cumulative_time_hr,window_cumulative_hr
0,160727,UDONTHANI,LMDCSO52Z,2025-07-10 00:00:00.969,กะเพราทะลุฟ้า (Kapraotalufha) -,À La Carte,LMF-250709-194339998,True,20.29,0.0,...,Daily Productivity,1,472,472,0,472,0,0,0.131111,0.000000
23335,160728,UDONTHANI,LMDCSO52Z,2025-07-10 10:59:50.639,Five Star (ห้าดาว) ดูโฮม อุดรธานี,Barbeque/Grill,LMF-250710-419574625,False,24.00,5.0,...,Daily Productivity,1,0,0,10,472,0,0,0.131111,0.000000
26735,160729,UDONTHANI,LMDCSO52Z,2025-07-10 11:16:01.800,ครัวพุงกาง (Pung Kang Kitchen),À La Carte,LMF-250710-984283443,True,19.00,5.0,...,Daily Productivity,2,981,981,11,1453,981,981,0.403611,0.272500
28080,160730,UDONTHANI,LMDCSO52Z,2025-07-10 11:22:21.890,ตำแซ่บอุดรธานี 1 (หน้าราชภัฏอุดร) (Tam Zaab Ud...,North East,LMF-250710-802551523,False,43.92,5.0,...,Daily Productivity,2,1355,0,11,1453,0,981,0.403611,0.272500
28248,160731,UDONTHANI,LMDCSO52Z,2025-07-10 11:23:11.190,อาหารตามสั่งครัวออโต้ (กะเพราแท้) (Auto Kitche...,À La Carte,LMF-250710-730251463,True,19.00,5.0,...,Daily Productivity,3,229,229,11,1682,229,1210,0.467222,0.336111
28379,160732,UDONTHANI,LMDCSO52Z,2025-07-10 11:23:44.760,MK Restaurants (เอ็มเค เรสโตรองต์) UD Town อุด...,Sukiyaki/Shabu,LMF-250710-389837582,False,32.34,10.0,...,Daily Productivity,3,675,0,11,1682,0,1210,0.467222,0.336111
28739,160733,UDONTHANI,LMDCSO52Z,2025-07-10 11:25:09.553,Mister Donut (มิสเตอร์ โดนัท) เซ็นทรัล พลาซ่า ...,Bakery/Cake,LMF-250710-869422156,False,27.89,10.0,...,Daily Productivity,3,1384,0,11,1682,0,1210,0.467222,0.336111
29227,160734,UDONTHANI,LMDCSO52Z,2025-07-10 11:27:28.911,❤️( ขนมจีบ ซาลาเปา ) ❤️เฮง เฮง ติ่มซำ❤️ (Heng ...,Dim Sum,LMF-250710-008052974,False,19.00,5.0,...,Daily Productivity,3,778,0,11,1682,0,1210,0.467222,0.336111
30631,160735,UDONTHANI,LMDCSO52Z,2025-07-10 11:34:18.485,ครัวคุณยาย (KRUA KHUN YAY) ครัวคุณยาย,À La Carte,LMF-250710-152003343,False,19.00,10.0,...,Daily Productivity,3,686,0,11,1682,0,1210,0.467222,0.336111
31276,160736,UDONTHANI,LMDCSO52Z,2025-07-10 11:37:23.135,คำผัด (เดลิเวอรี่) หน้าหอพักดีเฮ้าส์,À La Carte,LMF-250710-344254779,True,19.00,10.0,...,Daily Productivity,4,1383,1383,11,3065,1383,2593,0.851389,0.720278
